# Lab 07-07 · Frequency Response and Bode Plots

Companion to chapter [07-07](../../chapters/07-07-frequency-response-and-bode-plots.md). Handout: [README.md](README.md).

Every response below is computed by evaluating the transfer function or the state equations at $s = j\omega$, and every rise time by integrating a step response, so the chapter's asymptotes and formulas are being checked, not assumed. Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import math
import shutil
import subprocess
import numpy as np
from scipy.integrate import quad, solve_ivp
from scipy.optimize import brentq, minimize_scalar
from scipy.special import erfinv, sici
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HAVE_NGSPICE = shutil.which("ngspice") is not None
print("ngspice available:", HAVE_NGSPICE)
dB = lambda z: 20 * np.log10(np.abs(z))
deg = lambda z: np.degrees(np.angle(z))


def ngspice_values(path):
    out = subprocess.run(["ngspice", "-b", path], capture_output=True, text=True).stdout
    vals = {}
    for line in out.splitlines():
        parts = line.split("=")
        if len(parts) >= 2:
            try:
                vals[parts[0].strip()] = float(parts[1].split()[0])
            except (ValueError, IndexError):
                pass
    return vals


def bode_from_factors(f, K=1.0, zeros=(), poles=(), origin=0):
    # H(jf) and its straight-line magnitude asymptote, for real LHP zeros/poles given as corner frequencies in Hz
    s = 1j * f
    H = K * s**origin * np.ones_like(f, dtype=complex)
    asym = 20 * np.log10(abs(K)) + 20 * origin * np.log10(f) * np.ones_like(f)
    for z in zeros:
        H *= 1 + s / z
        asym += np.where(f > z, 20 * np.log10(f / z), 0.0)
    for p in poles:
        H /= 1 + s / p
        asym -= np.where(f > p, 20 * np.log10(f / p), 0.0)
    return H, asym

ngspice available: True


## Part 1 — Asymptotes and their errors

Example 07-07.2: $H = 100\,(jf/f_L)/\big((1 + jf/f_L)(1 + jf/f_H)\big)$ with $f_L$ = 15.92 Hz and $f_H$ = 15.92 kHz. In the helper the zero at the origin with gain $100/f_L$ gives the low-frequency asymptote.

**🔮 Predict:** the gain at 1.592 Hz and at 159.2 kHz.

In [2]:
fL = 1 / (2 * math.pi * 100e3 * 100e-9)
fH = 1 / (2 * math.pi * 1e3 * 10e-9)
f = np.logspace(-1, 6, 2001)
H, asym = bode_from_factors(f, K=100 / fL, poles=(fL, fH), origin=1)
for fx in (fL / 10, fL, 1e3, fH, 10 * fH):
    Hx, ax_ = bode_from_factors(np.array([fx]), K=100 / fL, poles=(fL, fH), origin=1)
    print(f"{fx:10.4g} Hz: exact {dB(Hx[0]):7.3f} dB, asymptote {ax_[0]:7.3f} dB, phase {deg(Hx[0]):8.3f} deg")
fig, (a1, a2) = plt.subplots(2, 1, figsize=(6, 4.5), sharex=True)
a1.semilogx(f, dB(H), label="exact"); a1.semilogx(f, asym, "--", label="asymptotes"); a1.legend(); a1.set_ylabel("dB")
a2.semilogx(f, deg(H)); a2.set_ylabel("deg"); a2.set_xlabel("Hz")
plt.close(fig)
assert abs(dB(bode_from_factors(np.array([fL]), K=100 / fL, poles=(fL, fH), origin=1)[0][0]) - 36.99) < 0.005

     1.592 Hz: exact  19.957 dB, asymptote  20.000 dB, phase   84.284 deg
     15.92 Hz: exact  36.990 dB, asymptote  40.000 dB, phase   44.943 deg
      1000 Hz: exact  39.982 dB, asymptote  40.000 dB, phase   -2.683 deg
 1.592e+04 Hz: exact  36.990 dB, asymptote  40.000 dB, phase  -44.943 deg
 1.592e+05 Hz: exact  19.957 dB, asymptote  20.000 dB, phase  -84.284 deg


In [3]:
if HAVE_NGSPICE:
    ng = ngspice_values("ex2_ac_amp.cir")
    print("ngspice:", {k: ng[k] for k in ("g_fl", "p_fl", "g_1k", "g_100k", "f_lo", "f_hi")})
    assert abs(ng["g_fl"] - 36.99) < 0.005 and abs(ng["f_hi"] - 15915) < 30

ngspice: {'g_fl': 36.9897, 'p_fl': 44.9427, 'g_1k': 39.98179, 'g_100k': 23.92776, 'f_lo': 15.91552, 'f_hi': 15915.47}


## Part 2 — A filter that amplifies

Example 07-07.3: the LC supply filter of 07-03 ($L$ = 4.7 µH, $R_s$ = 30 mΩ, $C$ = 22 µF) with its load $R_L$, from state equations $\mathbf{x} = (i_L, v)$.

**🔮 Predict:** the peak gain with no load at all, where only the 30 mΩ series resistance damps the tank.

In [4]:
L, Rs, Cf = 4.7e-6, 30e-3, 22e-6


def lc_H(f, RL=None, damp=None):
    gL = 0.0 if RL is None else 1 / RL
    if damp is None:
        A = np.array([[-Rs / L, -1 / L], [1 / Cf, -gL / Cf]])
        B = np.array([1 / L, 0]); Cm = np.array([0, 1])
    else:
        Rd, Cd = damp
        A = np.array([[-Rs / L, -1 / L, 0], [1 / Cf, -gL / Cf - 1 / (Rd * Cf), 1 / (Rd * Cf)], [0, 1 / (Rd * Cd), -1 / (Rd * Cd)]])
        B = np.array([1 / L, 0, 0]); Cm = np.array([0, 1, 0])
    n = A.shape[0]
    return np.array([Cm @ np.linalg.solve(2j * math.pi * fx * np.eye(n) - A, B) for fx in np.atleast_1d(f)])


def peak(**kw):
    r = minimize_scalar(lambda lf: -abs(lc_H(10**lf, **kw)[0]), bounds=(3, 5), method="bounded", options={"xatol": 1e-12})
    return dB(lc_H(10**r.x, **kw)[0]), 10**r.x


for lab, kw in (("10 ohm", {"RL": 10}), ("100 ohm", {"RL": 100}), ("no load", {}),
                ("10 ohm + damper", {"RL": 10, "damp": (math.sqrt(L / Cf), 4 * Cf)})):
    pk, fpk = peak(**kw)
    print(f"{lab:16s}: peak {pk:6.2f} dB at {fpk / 1e3:6.2f} kHz; at 500 kHz {dB(lc_H(500e3, **kw)[0]):7.2f} dB")
assert abs(peak(RL=10)[0] - 19.08) < 0.005
assert abs(peak(RL=10, damp=(math.sqrt(L / Cf), 4 * Cf))[0] - 2.27) < 0.005

10 ohm          : peak  19.08 dB at  15.63 kHz; at 500 kHz  -60.17 dB
100 ohm         : peak  23.16 dB at  15.64 kHz; at 500 kHz  -60.17 dB
no load         : peak  23.76 dB at  15.64 kHz; at 500 kHz  -60.17 dB
10 ohm + damper : peak   2.27 dB at   8.67 kHz; at 500 kHz  -60.17 dB


## Part 3 — Gain and phase are linked (Bode's integral)

For a minimum-phase $H$, the phase at $\omega_0$ is $\frac1\pi\int \frac{dA}{du}\ln\coth\frac{|u|}{2}\,du$ with $A = \ln|H|$ and $u = \ln(\omega/\omega_0)$ (chapter §3.6).

**🔮 Predict:** the phase of a response whose magnitude falls at a steady −30 dB/decade.

In [5]:
def bode_phase(Hfun, w0):
    def dAdu(u):
        h = 1e-5
        return (math.log(abs(Hfun(1j * w0 * math.exp(u + h)))) - math.log(abs(Hfun(1j * w0 * math.exp(u - h))))) / (2 * h)
    g = lambda u: dAdu(u) * math.log(1 / math.tanh(abs(u) / 2)) if u != 0 else 0.0
    return math.degrees((quad(g, -40, 0, limit=400)[0] + quad(g, 0, 40, limit=400)[0]) / math.pi)


print("-30 dB/decade:", round(bode_phase(lambda s: s**-1.5, 1.0), 3), "deg")
wp, wz = 2 * math.pi * 1e6, 2 * math.pi * 10e6
HA = lambda s: (1 + s / wz) / (1 + s / wp)
HB = lambda s: (1 - s / wz) / (1 + s / wp)
for fx in (1e6, 10e6, 100e6):
    w = 2 * math.pi * fx
    print(f"{fx:8.0e} Hz: from |H| {bode_phase(HA, w):8.2f}; actual A {deg(HA(1j * w)):8.2f}; actual B {deg(HB(1j * w)):8.2f}")
assert abs(bode_phase(HA, 2 * math.pi * 100e6) - deg(HA(2j * math.pi * 100e6))) < 0.05

-30 dB/decade: -135.0 deg
   1e+06 Hz: from |H|   -39.29; actual A   -39.29; actual B   -50.71
   1e+07 Hz: from |H|   -39.29; actual A   -39.29; actual B  -129.29
   1e+08 Hz: from |H|    -5.14; actual A    -5.14; actual B  -173.72


## Part 4 — Gain versus bandwidth, and phase margin

A dominant-pole op-amp, $A = A_0/(1 + jf/f_a)$ with $A_0 f_a$ = 10 MHz, in a loop with feedback fraction β. Then the chapter's guided exercise: add a second pole at 10 MHz and close a unity-gain loop.

**🔮 Predict:** the phase margin, the peaking in dB and the frequency of the peak.

In [6]:
GBW, A0 = 10e6, 10 ** (130 / 20)
fa = GBW / A0
A = lambda f, f2=None: A0 / (1 + 1j * f / fa) / (1 if f2 is None else (1 + 1j * f / f2))
for gain in (1, 10, 100, 1000):
    beta = 1 / gain
    Hcl = lambda f: A(f) / (1 + A(f) * beta)
    g0 = abs(Hcl(1.0))
    bw = brentq(lambda f: abs(Hcl(f)) - g0 / math.sqrt(2), 1, 1e9)
    print(f"gain {gain:5d}: bandwidth {bw / 1e3:9.1f} kHz, gain x bandwidth {gain * bw / 1e6:6.3f} MHz")
f2 = 10e6
T = lambda f: A(f, f2)
fc = brentq(lambda f: abs(T(f)) - 1, 1e5, 1e9)
pm = 180 + deg(T(fc))
Hb = lambda f: T(f) / (1 + T(f))
r = minimize_scalar(lambda lf: -abs(Hb(10**lf)), bounds=(5, 8), method="bounded", options={"xatol": 1e-12})
print(f"crossover {fc / 1e6:.3f} MHz, phase margin {pm:.2f} deg, peak {dB(Hb(10**r.x)):.3f} dB at {10**r.x / 1e6:.3f} MHz")
assert abs(pm - 51.83) < 0.01 and abs(dB(Hb(10**r.x)) - 1.249) < 0.001
if HAVE_NGSPICE:
    ng = ngspice_values("ex5_opamp.cir")
    print("ngspice:", {k: ng.get(k) for k in ("bw_t", "bw_inv", "bw_cas", "pm", "pk_buf", "fpk_buf")})

gain     1: bandwidth   10000.0 kHz, gain x bandwidth 10.000 MHz
gain    10: bandwidth    1000.0 kHz, gain x bandwidth 10.000 MHz
gain   100: bandwidth     100.0 kHz, gain x bandwidth 10.000 MHz
gain  1000: bandwidth      10.0 kHz, gain x bandwidth 10.003 MHz
crossover 7.862 MHz, phase margin 51.83 deg, peak 1.249 dB at 7.071 MHz


ngspice: {'bw_t': 100003.1, 'bw_inv': 99012.94, 'bw_cas': 643589.6, 'pm': 51.82728, 'pk_buf': 1.249391, 'fpk_buf': 7071312.0}


## Part 5 — Rise time and bandwidth

Three responses with the same −3 dB bandwidth: a single pole, a Gaussian and an ideal brick-wall low-pass.

**🔮 Predict:** which rises fastest for its bandwidth?

In [7]:
# single pole: integrate the step response
tau = 1.0
sol = solve_ivp(lambda t, x: (1 - x) / tau, (0, 20), [0.0], dense_output=True, rtol=1e-11, atol=1e-13)
t = np.linspace(0, 20, 200001); y = sol.sol(t)[0]
k_single = (np.interp(0.9, y, t) - np.interp(0.1, y, t)) / (2 * math.pi * tau)
# Gaussian impulse response of standard deviation 1: |H(f)| = exp(-2 pi^2 f^2)
f3g = math.sqrt(math.log(2) / 2) / (math.sqrt(2) * math.pi)
k_gauss = 2 * math.sqrt(2) * erfinv(0.8) * f3g
# brick wall: step = 1/2 + Si(2 pi fc t)/pi
s = lambda x: 0.5 + sici(x)[0] / math.pi
k_brick = (brentq(lambda x: s(x) - 0.9, 0, 5) - brentq(lambda x: s(x) - 0.1, -5, 0)) / (2 * math.pi)
print(f"t_r x BW: Gaussian {k_gauss:.4f}, single pole {k_single:.4f}, brick wall {k_brick:.4f}")
assert abs(k_single - 0.3497) < 1e-4 and abs(k_brick - 0.4458) < 1e-4
if HAVE_NGSPICE:
    ng = ngspice_values("ex6_rise.cir")
    print("ngspice:", ng.get("tr_scope"), ng.get("tr_sys"))

t_r x BW: Gaussian 0.3396, single pole 0.3497, brick wall 0.4458


ngspice: 1.748496e-09 2.138951e-09


## Your turn

1. What is the peak gain of Example 07-07.3's filter with no load (only $R_s$ damps it)? Find ζ from $R_s/(2Z_0)$ first.
2. A minimum-phase response falls at a steady −30 dB/decade. What is its phase?
3. An op-amp with GBW = 10 MHz is used as an inverting amplifier of gain −3. What is its bandwidth?

<details><summary>Answers</summary>

1. $Z_0 = \sqrt{L/C}$ = 0.4622 Ω, ζ = 0.03/(2 × 0.4622) = 0.03245, and the peak is $1/(2\zeta\sqrt{1 - \zeta^2})$ = 15.42, or **23.76 dB**, higher than any loaded case.
2. $k = -1.5$, so **−135°**.
3. The noise gain is 1 + 3 = 4, so **2.5 MHz**.

</details>

In [8]:
Z0 = math.sqrt(L / Cf); z = Rs / (2 * Z0)
print(f"1) {20 * math.log10(1 / (2 * z * math.sqrt(1 - z * z))):.2f} dB (formula); {peak()[0]:.2f} dB (numerical)")
print(f"2) {bode_phase(lambda s: s**-1.5, 1.0):.1f} deg")
print(f"3) {GBW / 4 / 1e6:.2f} MHz")
assert abs(20 * math.log10(1 / (2 * z * math.sqrt(1 - z * z))) - 23.76) < 0.005 and abs(peak()[0] - 23.76) < 0.005

1) 23.76 dB (formula); 23.76 dB (numerical)
2) -135.0 deg
3) 2.50 MHz
